# Exploratory Data Analysis: Pitch Type & Location Prediction

This notebook performs comprehensive EDA on MLB pitch data to identify features that predict:
1. **Pitch Type** - What pitch will be thrown next (FF, SL, CH, CU, etc.)
2. **Pitch Location** - Where will the pitch be located (px, pz coordinates)

## Table of Contents
1. [Setup & Data Loading](#setup)
2. [Phase 1: Data Quality & Distribution Analysis](#phase1)
3. [Phase 2: Feature-Target Relationship Analysis](#phase2)
4. [Phase 3: Pitcher & Batter Analysis](#phase3)
5. [Phase 4: Physical Metrics Analysis](#phase4)
6. [Phase 5: Feature Importance & Correlation](#phase5)
7. [Conclusions & Recommendations](#conclusions)

## 1. Setup & Data Loading <a id='setup'></a>

In [ ]:
# Core imports
import polars as pl
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Rectangle
from matplotlib.colors import LinearSegmentedColormap

# Statistics
from scipy import stats
from sklearn.feature_selection import mutual_info_classif, mutual_info_regression
from sklearn.preprocessing import LabelEncoder

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11

# Data path
DATA_PATH = Path('../data/processed/livefeeds')

print('Setup complete!')

In [ ]:
# Pitch type mappings
PITCH_TYPE_CODES = {
    'FF': 'Four-Seam Fastball',
    'SI': 'Sinker',
    'FC': 'Cutter',
    'CH': 'Changeup',
    'SL': 'Slider',
    'CU': 'Curveball',
    'KC': 'Knuckle Curve',
    'ST': 'Sweeper',
    'FS': 'Splitter',
    'KN': 'Knuckleball',
}

# Strike zone dimensions (in feet)
STRIKE_ZONE = {
    'left': -0.83,   # ~10 inches from center
    'right': 0.83,
    'bottom': 1.5,   # Typical bottom
    'top': 3.5       # Typical top
}

def draw_strike_zone(ax):
    """Draw strike zone rectangle on a matplotlib axis."""
    zone = Rectangle(
        (STRIKE_ZONE['left'], STRIKE_ZONE['bottom']),
        STRIKE_ZONE['right'] - STRIKE_ZONE['left'],
        STRIKE_ZONE['top'] - STRIKE_ZONE['bottom'],
        fill=False, edgecolor='black', linewidth=2
    )
    ax.add_patch(zone)
    ax.set_xlim(-2.5, 2.5)
    ax.set_ylim(0, 5)
    ax.set_xlabel('Horizontal Position (ft)')
    ax.set_ylabel('Vertical Position (ft)')
    ax.set_aspect('equal')

In [ ]:
# Load data - using 2024 season for initial exploration
print('Loading 2024 season data...')
df = pl.scan_parquet(str(DATA_PATH / '2024' / '*.parquet')).collect()
print(f'Loaded {len(df):,} pitches from {df["game_pk"].n_unique():,} games')
print(f'Columns: {len(df.columns)}')

In [ ]:
# Preview the data
df.head()

In [ ]:
# List all columns
print('All columns:')
for i, col in enumerate(sorted(df.columns)):
    print(f'{i+1:2}. {col}')

---
## 2. Phase 1: Data Quality & Distribution Analysis <a id='phase1'></a>

### 1.1 Missing Value Analysis

In [ ]:
# Calculate missing values for each column
missing_stats = []
for col in df.columns:
    null_count = df[col].null_count()
    null_pct = null_count / len(df) * 100
    missing_stats.append({
        'column': col,
        'null_count': null_count,
        'null_pct': null_pct,
        'dtype': str(df[col].dtype)
    })

missing_df = pl.DataFrame(missing_stats).sort('null_pct', descending=True)

# Display columns with missing values
print('Columns with missing values (>0%):')
print(missing_df.filter(pl.col('null_pct') > 0))

In [ ]:
# Visualize missing value patterns
missing_cols = missing_df.filter(pl.col('null_pct') > 0).sort('null_pct', descending=True)

if len(missing_cols) > 0:
    fig, ax = plt.subplots(figsize=(12, max(6, len(missing_cols) * 0.3)))
    
    cols = missing_cols['column'].to_list()[:30]  # Top 30
    pcts = missing_cols['null_pct'].to_list()[:30]
    
    colors = ['red' if p > 50 else 'orange' if p > 10 else 'green' for p in pcts]
    ax.barh(cols, pcts, color=colors)
    ax.set_xlabel('Missing %')
    ax.set_title('Missing Values by Column')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print('No missing values found!')

### 1.2 Target Variable Distributions

In [ ]:
# Pitch type distribution
pitch_counts = df.group_by('pitch_type_code').agg(
    pl.count().alias('count')
).sort('count', descending=True)

# Add percentage
total = pitch_counts['count'].sum()
pitch_counts = pitch_counts.with_columns(
    (pl.col('count') / total * 100).round(2).alias('pct')
)

print('Pitch Type Distribution:')
print(pitch_counts)

In [ ]:
# Visualize pitch type distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Bar chart
pitch_types = pitch_counts['pitch_type_code'].to_list()
counts = pitch_counts['count'].to_list()
pcts = pitch_counts['pct'].to_list()

colors = plt.cm.Set3(np.linspace(0, 1, len(pitch_types)))
bars = axes[0].bar(pitch_types, counts, color=colors)
axes[0].set_xlabel('Pitch Type')
axes[0].set_ylabel('Count')
axes[0].set_title('Pitch Type Distribution')
axes[0].tick_params(axis='x', rotation=45)

# Add percentage labels
for bar, pct in zip(bars, pcts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                 f'{pct:.1f}%', ha='center', va='bottom', fontsize=9)

# Pie chart (top 8 + other)
top_8 = pitch_types[:8]
top_8_pcts = pcts[:8]
other_pct = sum(pcts[8:])
if other_pct > 0:
    top_8.append('Other')
    top_8_pcts.append(other_pct)

axes[1].pie(top_8_pcts, labels=top_8, autopct='%1.1f%%', colors=colors[:len(top_8)])
axes[1].set_title('Pitch Type Share')

plt.tight_layout()
plt.show()

In [ ]:
# Location distribution (px, pz)
# Filter out null locations
loc_df = df.filter(pl.col('px').is_not_null() & pl.col('pz').is_not_null())

print(f'Pitches with valid locations: {len(loc_df):,} ({len(loc_df)/len(df)*100:.1f}%)')
print(f'\npx (horizontal): min={loc_df["px"].min():.2f}, max={loc_df["px"].max():.2f}, mean={loc_df["px"].mean():.2f}')
print(f'pz (vertical): min={loc_df["pz"].min():.2f}, max={loc_df["pz"].max():.2f}, mean={loc_df["pz"].mean():.2f}')

In [ ]:
# Location heatmap
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Sample for plotting (to speed up)
sample = loc_df.sample(min(50000, len(loc_df)), seed=42)

# Scatter plot
axes[0].scatter(sample['px'].to_list(), sample['pz'].to_list(), alpha=0.1, s=1)
draw_strike_zone(axes[0])
axes[0].set_title('Pitch Location Distribution')

# 2D histogram (heatmap)
h = axes[1].hist2d(
    loc_df['px'].to_list(), 
    loc_df['pz'].to_list(),
    bins=50, 
    range=[[-2.5, 2.5], [0, 5]],
    cmap='YlOrRd'
)
plt.colorbar(h[3], ax=axes[1], label='Count')
draw_strike_zone(axes[1])
axes[1].set_title('Pitch Location Density')

plt.tight_layout()
plt.show()

### 1.3 Temporal Patterns (Multi-Season)

In [ ]:
# Load all seasons for temporal analysis
print('Loading all seasons for temporal analysis...')
all_seasons = []

for season in ['2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']:
    season_path = DATA_PATH / season
    if season_path.exists():
        season_df = pl.scan_parquet(str(season_path / '*.parquet')).select(
            ['pitch_type_code', 'pitch_start_speed', 'px', 'pz']
        ).collect()
        season_df = season_df.with_columns(pl.lit(season).alias('season'))
        all_seasons.append(season_df)
        print(f'  {season}: {len(season_df):,} pitches')

multi_season_df = pl.concat(all_seasons)
print(f'\nTotal: {len(multi_season_df):,} pitches')

In [ ]:
# Pitch type trends by season
pitch_by_season = multi_season_df.group_by(['season', 'pitch_type_code']).agg(
    pl.count().alias('count')
)

# Calculate percentage within each season
season_totals = pitch_by_season.group_by('season').agg(pl.col('count').sum().alias('total'))
pitch_by_season = pitch_by_season.join(season_totals, on='season')
pitch_by_season = pitch_by_season.with_columns(
    (pl.col('count') / pl.col('total') * 100).alias('pct')
)

# Pivot for plotting
pitch_pivot = pitch_by_season.pivot(
    values='pct',
    index='season',
    on='pitch_type_code'
).sort('season')

print('Pitch Type Usage by Season (%):')
print(pitch_pivot)

In [ ]:
# Visualize pitch type trends
fig, ax = plt.subplots(figsize=(14, 8))

# Select key pitch types to track
key_pitches = ['FF', 'SI', 'SL', 'CH', 'CU', 'FC', 'ST']
seasons = pitch_pivot['season'].to_list()

for pitch in key_pitches:
    if pitch in pitch_pivot.columns:
        values = pitch_pivot[pitch].to_list()
        ax.plot(seasons, values, marker='o', label=pitch, linewidth=2)

ax.set_xlabel('Season')
ax.set_ylabel('Usage %')
ax.set_title('Pitch Type Usage Trends by Season')
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Velocity trends by season
velocity_by_season = multi_season_df.filter(
    pl.col('pitch_start_speed').is_not_null()
).group_by('season').agg(
    pl.col('pitch_start_speed').mean().alias('avg_velocity'),
    pl.col('pitch_start_speed').std().alias('std_velocity'),
    pl.col('pitch_start_speed').max().alias('max_velocity')
).sort('season')

print('Average Velocity by Season:')
print(velocity_by_season)

---
## 3. Phase 2: Feature-Target Relationship Analysis <a id='phase2'></a>

### 2.1 Count Impact on Pitch Selection

In [ ]:
# Parse count into balls and strikes
def parse_count(count_str):
    if not count_str or '-' not in str(count_str):
        return None, None
    parts = str(count_str).split('-')
    return int(parts[0]), int(parts[1])

# Add count columns
df_with_count = df.with_columns([
    pl.col('count_after_pitch').map_elements(
        lambda x: parse_count(x)[0], return_dtype=pl.Int64
    ).alias('balls'),
    pl.col('count_after_pitch').map_elements(
        lambda x: parse_count(x)[1], return_dtype=pl.Int64
    ).alias('strikes')
])

# Filter valid counts
df_with_count = df_with_count.filter(
    pl.col('balls').is_not_null() & pl.col('strikes').is_not_null()
)

print(f'Pitches with valid counts: {len(df_with_count):,}')

In [ ]:
# Pitch type distribution by count
count_pitch = df_with_count.group_by(['balls', 'strikes', 'pitch_type_code']).agg(
    pl.count().alias('count')
)

# Calculate percentage within each count state
count_totals = count_pitch.group_by(['balls', 'strikes']).agg(pl.col('count').sum().alias('total'))
count_pitch = count_pitch.join(count_totals, on=['balls', 'strikes'])
count_pitch = count_pitch.with_columns(
    (pl.col('count') / pl.col('total') * 100).round(1).alias('pct')
)

# Create count label
count_pitch = count_pitch.with_columns(
    (pl.col('balls').cast(str) + '-' + pl.col('strikes').cast(str)).alias('count_label')
)

In [ ]:
# Heatmap: Fastball usage by count
ff_by_count = count_pitch.filter(pl.col('pitch_type_code') == 'FF')

# Pivot for heatmap
ff_pivot = ff_by_count.pivot(
    values='pct',
    index='strikes',
    on='balls'
).sort('strikes')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# FF heatmap
ff_data = ff_pivot.select(['0', '1', '2', '3']).to_numpy()
im = axes[0].imshow(ff_data, cmap='YlOrRd', aspect='auto')
axes[0].set_xticks([0, 1, 2, 3])
axes[0].set_xticklabels(['0', '1', '2', '3'])
axes[0].set_yticks([0, 1, 2])
axes[0].set_yticklabels(['0', '1', '2'])
axes[0].set_xlabel('Balls')
axes[0].set_ylabel('Strikes')
axes[0].set_title('Four-Seam Fastball Usage by Count (%)')
plt.colorbar(im, ax=axes[0])

# Add text annotations
for i in range(3):
    for j in range(4):
        if j < len(ff_data[i]) and not np.isnan(ff_data[i][j]):
            axes[0].text(j, i, f'{ff_data[i][j]:.1f}', ha='center', va='center', fontsize=12)

# Breaking ball (SL + CH + CU) by count
breaking_by_count = count_pitch.filter(
    pl.col('pitch_type_code').is_in(['SL', 'CH', 'CU', 'ST'])
).group_by(['balls', 'strikes']).agg(
    pl.col('pct').sum().alias('pct')
)

breaking_pivot = breaking_by_count.pivot(
    values='pct',
    index='strikes',
    on='balls'
).sort('strikes')

breaking_data = breaking_pivot.select(['0', '1', '2', '3']).to_numpy()
im2 = axes[1].imshow(breaking_data, cmap='YlGnBu', aspect='auto')
axes[1].set_xticks([0, 1, 2, 3])
axes[1].set_xticklabels(['0', '1', '2', '3'])
axes[1].set_yticks([0, 1, 2])
axes[1].set_yticklabels(['0', '1', '2'])
axes[1].set_xlabel('Balls')
axes[1].set_ylabel('Strikes')
axes[1].set_title('Breaking Ball Usage by Count (%)')
plt.colorbar(im2, ax=axes[1])

for i in range(3):
    for j in range(4):
        if j < len(breaking_data[i]) and not np.isnan(breaking_data[i][j]):
            axes[1].text(j, i, f'{breaking_data[i][j]:.1f}', ha='center', va='center', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# Location by count state
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

count_states = [
    ('0-0', 'First Pitch'),
    ('0-2', 'Pitcher Ahead'),
    ('3-0', "Hitter's Count"),
    ('1-1', 'Even Count'),
    ('2-2', 'Two Strikes'),
    ('3-2', 'Full Count')
]

loc_df_count = df_with_count.filter(
    pl.col('px').is_not_null() & pl.col('pz').is_not_null()
)

for idx, (count, title) in enumerate(count_states):
    ax = axes[idx // 3, idx % 3]
    balls, strikes = int(count[0]), int(count[2])
    
    count_data = loc_df_count.filter(
        (pl.col('balls') == balls) & (pl.col('strikes') == strikes)
    )
    
    if len(count_data) > 0:
        ax.hist2d(
            count_data['px'].to_list(),
            count_data['pz'].to_list(),
            bins=30,
            range=[[-2.5, 2.5], [0, 5]],
            cmap='YlOrRd'
        )
        draw_strike_zone(ax)
        ax.set_title(f'{count} - {title}\n(n={len(count_data):,})')

plt.tight_layout()
plt.show()

### 2.2 Handedness Effects

In [ ]:
# Pitch type by pitcher handedness
pitch_by_hand = df.group_by(['throw_side', 'pitch_type_code']).agg(
    pl.count().alias('count')
)

hand_totals = pitch_by_hand.group_by('throw_side').agg(pl.col('count').sum().alias('total'))
pitch_by_hand = pitch_by_hand.join(hand_totals, on='throw_side')
pitch_by_hand = pitch_by_hand.with_columns(
    (pl.col('count') / pl.col('total') * 100).round(1).alias('pct')
)

# Pivot for comparison
hand_pivot = pitch_by_hand.pivot(
    values='pct',
    index='pitch_type_code',
    on='throw_side'
)

print('Pitch Type by Pitcher Handedness (%):')
print(hand_pivot.sort('L', descending=True))

In [ ]:
# Visualize handedness differences
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Pitcher handedness
pitch_types = hand_pivot['pitch_type_code'].to_list()
l_pct = hand_pivot['L'].to_list()
r_pct = hand_pivot['R'].to_list()

x = np.arange(len(pitch_types))
width = 0.35

axes[0].bar(x - width/2, l_pct, width, label='LHP', color='blue', alpha=0.7)
axes[0].bar(x + width/2, r_pct, width, label='RHP', color='red', alpha=0.7)
axes[0].set_xticks(x)
axes[0].set_xticklabels(pitch_types, rotation=45)
axes[0].set_ylabel('Usage %')
axes[0].set_title('Pitch Type by Pitcher Handedness')
axes[0].legend()

# Platoon matchups
platoon = df.filter(
    pl.col('throw_side').is_not_null() & pl.col('bat_side').is_not_null()
).with_columns(
    (pl.col('throw_side') + ' vs ' + pl.col('bat_side')).alias('matchup')
)

matchup_counts = platoon.group_by('matchup').agg(pl.count().alias('count'))
total = matchup_counts['count'].sum()
matchup_counts = matchup_counts.with_columns(
    (pl.col('count') / total * 100).round(1).alias('pct')
).sort('count', descending=True)

axes[1].bar(matchup_counts['matchup'].to_list(), matchup_counts['pct'].to_list())
axes[1].set_ylabel('% of Pitches')
axes[1].set_title('Platoon Matchup Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# Location heatmaps by platoon matchup
fig, axes = plt.subplots(2, 2, figsize=(12, 12))

matchups = ['R vs R', 'R vs L', 'L vs R', 'L vs L']
loc_platoon = platoon.filter(pl.col('px').is_not_null() & pl.col('pz').is_not_null())

for idx, matchup in enumerate(matchups):
    ax = axes[idx // 2, idx % 2]
    data = loc_platoon.filter(pl.col('matchup') == matchup)
    
    if len(data) > 0:
        ax.hist2d(
            data['px'].to_list(),
            data['pz'].to_list(),
            bins=40,
            range=[[-2.5, 2.5], [0, 5]],
            cmap='YlOrRd'
        )
        draw_strike_zone(ax)
        ax.set_title(f'{matchup} (n={len(data):,})')

plt.suptitle('Pitch Location by Platoon Matchup', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### 2.3 Game Situation Impact

In [ ]:
# Runners on base effect
df_runners = df.with_columns([
    pl.col('is_runner_on_first').fill_null(False).alias('runner_1b'),
    pl.col('is_runner_on_second').fill_null(False).alias('runner_2b'),
    pl.col('is_runner_on_third').fill_null(False).alias('runner_3b'),
])

# Create base state label
df_runners = df_runners.with_columns(
    pl.when(~pl.col('runner_1b') & ~pl.col('runner_2b') & ~pl.col('runner_3b'))
    .then(pl.lit('Bases Empty'))
    .when(pl.col('runner_1b') & pl.col('runner_2b') & pl.col('runner_3b'))
    .then(pl.lit('Bases Loaded'))
    .when(pl.col('runner_1b') & ~pl.col('runner_2b') & ~pl.col('runner_3b'))
    .then(pl.lit('Runner on 1st'))
    .when(~pl.col('runner_1b') & pl.col('runner_2b') & ~pl.col('runner_3b'))
    .then(pl.lit('Runner on 2nd'))
    .when(~pl.col('runner_1b') & ~pl.col('runner_2b') & pl.col('runner_3b'))
    .then(pl.lit('Runner on 3rd'))
    .otherwise(pl.lit('Multiple Runners'))
    .alias('base_state')
)

# Pitch type by base state
pitch_by_bases = df_runners.group_by(['base_state', 'pitch_type_code']).agg(
    pl.count().alias('count')
)

base_totals = pitch_by_bases.group_by('base_state').agg(pl.col('count').sum().alias('total'))
pitch_by_bases = pitch_by_bases.join(base_totals, on='base_state')
pitch_by_bases = pitch_by_bases.with_columns(
    (pl.col('count') / pl.col('total') * 100).round(1).alias('pct')
)

# Show FF usage by base state
ff_by_bases = pitch_by_bases.filter(pl.col('pitch_type_code') == 'FF').sort('pct', descending=True)
print('Fastball Usage by Base State:')
print(ff_by_bases.select(['base_state', 'pct', 'total']))

In [ ]:
# Inning effects
pitch_by_inning = df.filter(pl.col('inning').is_not_null()).group_by(['inning', 'pitch_type_code']).agg(
    pl.count().alias('count')
)

inning_totals = pitch_by_inning.group_by('inning').agg(pl.col('count').sum().alias('total'))
pitch_by_inning = pitch_by_inning.join(inning_totals, on='inning')
pitch_by_inning = pitch_by_inning.with_columns(
    (pl.col('count') / pl.col('total') * 100).alias('pct')
)

# FF usage by inning
ff_by_inning = pitch_by_inning.filter(
    (pl.col('pitch_type_code') == 'FF') & (pl.col('inning') <= 9)
).sort('inning')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ff_by_inning['inning'].to_list(), ff_by_inning['pct'].to_list(), marker='o', linewidth=2)
ax.set_xlabel('Inning')
ax.set_ylabel('Fastball Usage %')
ax.set_title('Fastball Usage by Inning')
ax.set_xticks(range(1, 10))
ax.grid(True, alpha=0.3)
plt.show()

### 2.4 Sequencing Patterns

In [ ]:
# Pitch sequence analysis - what follows each pitch type?
# Sort by game and at-bat to ensure proper ordering
df_seq = df.sort(['game_pk', 'at_bat_index', 'pitch_number'])

# Add previous pitch type
df_seq = df_seq.with_columns(
    pl.col('pitch_type_code')
    .shift(1)
    .over(['game_pk', 'at_bat_index'])
    .alias('prev_pitch_type')
)

# Filter to pitches with previous pitch (pitch 2+)
df_seq = df_seq.filter(pl.col('prev_pitch_type').is_not_null())

# Create transition matrix
transitions = df_seq.group_by(['prev_pitch_type', 'pitch_type_code']).agg(
    pl.count().alias('count')
)

# Calculate row percentages
prev_totals = transitions.group_by('prev_pitch_type').agg(pl.col('count').sum().alias('total'))
transitions = transitions.join(prev_totals, on='prev_pitch_type')
transitions = transitions.with_columns(
    (pl.col('count') / pl.col('total') * 100).round(1).alias('pct')
)

print('Pitch Transition Counts:')
print(transitions.sort('count', descending=True).head(20))

In [ ]:
# Transition matrix heatmap
# Pivot for heatmap
main_pitches = ['FF', 'SI', 'SL', 'CH', 'CU', 'FC', 'ST']
trans_filtered = transitions.filter(
    pl.col('prev_pitch_type').is_in(main_pitches) &
    pl.col('pitch_type_code').is_in(main_pitches)
)

trans_pivot = trans_filtered.pivot(
    values='pct',
    index='prev_pitch_type',
    on='pitch_type_code'
)

# Reorder to match main_pitches
trans_matrix = []
for prev in main_pitches:
    row = trans_pivot.filter(pl.col('prev_pitch_type') == prev)
    if len(row) > 0:
        row_data = []
        for curr in main_pitches:
            if curr in row.columns:
                val = row[curr][0]
                row_data.append(val if val is not None else 0)
            else:
                row_data.append(0)
        trans_matrix.append(row_data)
    else:
        trans_matrix.append([0] * len(main_pitches))

trans_matrix = np.array(trans_matrix)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(trans_matrix, cmap='YlOrRd')
ax.set_xticks(range(len(main_pitches)))
ax.set_xticklabels(main_pitches)
ax.set_yticks(range(len(main_pitches)))
ax.set_yticklabels(main_pitches)
ax.set_xlabel('Current Pitch')
ax.set_ylabel('Previous Pitch')
ax.set_title('Pitch Transition Matrix (%)')
plt.colorbar(im, ax=ax)

# Add text annotations
for i in range(len(main_pitches)):
    for j in range(len(main_pitches)):
        val = trans_matrix[i, j]
        if val > 0:
            color = 'white' if val > 20 else 'black'
            ax.text(j, i, f'{val:.0f}', ha='center', va='center', fontsize=9, color=color)

plt.tight_layout()
plt.show()

---
## 4. Phase 3: Pitcher & Batter Analysis <a id='phase3'></a>

### 3.1 Pitcher Repertoire Clustering

In [ ]:
# Calculate pitch mix for each pitcher
pitcher_mix = df.group_by(['pitcher_id', 'pitch_type_code']).agg(
    pl.count().alias('count')
)

pitcher_totals = pitcher_mix.group_by('pitcher_id').agg(pl.col('count').sum().alias('total'))
pitcher_mix = pitcher_mix.join(pitcher_totals, on='pitcher_id')
pitcher_mix = pitcher_mix.with_columns(
    (pl.col('count') / pl.col('total') * 100).alias('pct')
)

# Pivot to get pitch usage by pitcher
pitcher_pivot = pitcher_mix.pivot(
    values='pct',
    index='pitcher_id',
    on='pitch_type_code'
).fill_null(0)

# Filter to pitchers with enough pitches
pitcher_pivot = pitcher_pivot.join(
    pitcher_totals.filter(pl.col('total') >= 100),
    on='pitcher_id'
)

print(f'Pitchers with 100+ pitches: {len(pitcher_pivot)}')

In [ ]:
# Cluster pitchers by repertoire
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Use main pitch types for clustering
cluster_cols = ['FF', 'SI', 'SL', 'CH', 'CU', 'FC', 'ST']
available_cols = [c for c in cluster_cols if c in pitcher_pivot.columns]

X = pitcher_pivot.select(available_cols).to_numpy()
X = np.nan_to_num(X, 0)

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# K-means clustering
n_clusters = 5
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)

pitcher_pivot = pitcher_pivot.with_columns(pl.Series('cluster', clusters))

# Analyze clusters
cluster_means = pitcher_pivot.group_by('cluster').agg(
    [pl.col(c).mean().round(1).alias(c) for c in available_cols] +
    [pl.count().alias('n_pitchers')]
).sort('cluster')

print('Pitcher Cluster Profiles (Average Pitch Usage %):')
print(cluster_means)

In [ ]:
# Visualize cluster profiles
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(available_cols))
width = 0.15

for i in range(n_clusters):
    cluster_data = cluster_means.filter(pl.col('cluster') == i)
    values = [cluster_data[c][0] for c in available_cols]
    n = cluster_data['n_pitchers'][0]
    ax.bar(x + i * width, values, width, label=f'Cluster {i} (n={n})')

ax.set_xticks(x + width * (n_clusters - 1) / 2)
ax.set_xticklabels(available_cols)
ax.set_ylabel('Average Usage %')
ax.set_title('Pitcher Repertoire Clusters')
ax.legend()
plt.tight_layout()
plt.show()

### 3.2 Pitcher Location Tendencies

In [ ]:
# Pitcher location consistency
pitcher_loc = df.filter(
    pl.col('px').is_not_null() & pl.col('pz').is_not_null()
).group_by('pitcher_id').agg(
    pl.col('px').mean().alias('mean_px'),
    pl.col('pz').mean().alias('mean_pz'),
    pl.col('px').std().alias('std_px'),
    pl.col('pz').std().alias('std_pz'),
    pl.count().alias('n_pitches')
).filter(pl.col('n_pitches') >= 100)

# Add location scatter (combined std)
pitcher_loc = pitcher_loc.with_columns(
    (pl.col('std_px') + pl.col('std_pz')).alias('location_scatter')
)

print('Pitcher Location Consistency (lower scatter = more consistent):')
print(pitcher_loc.sort('location_scatter').head(10))

---
## 5. Phase 4: Physical Metrics Analysis <a id='phase4'></a>

### 4.1 Pitch Movement Characteristics

In [ ]:
# Pitch movement by type
movement_df = df.filter(
    pl.col('pfxX').is_not_null() & pl.col('pfxZ').is_not_null()
).select(['pitch_type_code', 'pfxX', 'pfxZ', 'pitch_start_speed'])

print(f'Pitches with movement data: {len(movement_df):,}')

# Average movement by pitch type
movement_avg = movement_df.group_by('pitch_type_code').agg(
    pl.col('pfxX').mean().round(2).alias('avg_pfxX'),
    pl.col('pfxZ').mean().round(2).alias('avg_pfxZ'),
    pl.col('pitch_start_speed').mean().round(1).alias('avg_velo'),
    pl.count().alias('count')
).filter(pl.col('count') >= 1000).sort('avg_velo', descending=True)

print('\nPitch Movement and Velocity by Type:')
print(movement_avg)

In [ ]:
# Movement scatter plot by pitch type
fig, ax = plt.subplots(figsize=(12, 10))

pitch_types = movement_avg['pitch_type_code'].to_list()
colors = plt.cm.Set1(np.linspace(0, 1, len(pitch_types)))

for i, ptype in enumerate(pitch_types):
    type_data = movement_df.filter(pl.col('pitch_type_code') == ptype).sample(
        min(5000, len(movement_df.filter(pl.col('pitch_type_code') == ptype))), seed=42
    )
    ax.scatter(
        type_data['pfxX'].to_list(),
        type_data['pfxZ'].to_list(),
        alpha=0.3,
        s=10,
        label=ptype,
        color=colors[i]
    )

ax.axhline(y=0, color='black', linestyle='--', alpha=0.3)
ax.axvline(x=0, color='black', linestyle='--', alpha=0.3)
ax.set_xlabel('Horizontal Movement (pfxX, inches)')
ax.set_ylabel('Vertical Movement (pfxZ, inches)')
ax.set_title('Pitch Movement Profile by Type')
ax.legend(loc='upper left')
ax.set_xlim(-25, 25)
ax.set_ylim(-25, 25)
plt.tight_layout()
plt.show()

In [ ]:
# Velocity distribution by pitch type
fig, ax = plt.subplots(figsize=(14, 6))

velo_df = df.filter(pl.col('pitch_start_speed').is_not_null())

for ptype in ['FF', 'SI', 'FC', 'SL', 'CH', 'CU', 'ST']:
    type_velo = velo_df.filter(pl.col('pitch_type_code') == ptype)['pitch_start_speed'].to_list()
    if len(type_velo) > 0:
        ax.hist(type_velo, bins=50, alpha=0.5, label=ptype, density=True)

ax.set_xlabel('Velocity (mph)')
ax.set_ylabel('Density')
ax.set_title('Velocity Distribution by Pitch Type')
ax.legend()
plt.tight_layout()
plt.show()

### 4.2 Release Point Analysis

In [ ]:
# Release point by pitch type (for RHP)
release_df = df.filter(
    pl.col('x0').is_not_null() & 
    pl.col('z0').is_not_null() &
    (pl.col('throw_side') == 'R')  # Focus on RHP for cleaner visualization
)

release_avg = release_df.group_by('pitch_type_code').agg(
    pl.col('x0').mean().round(3).alias('avg_x0'),
    pl.col('z0').mean().round(3).alias('avg_z0'),
    pl.col('x0').std().round(3).alias('std_x0'),
    pl.col('z0').std().round(3).alias('std_z0'),
    pl.count().alias('count')
).filter(pl.col('count') >= 1000).sort('avg_z0', descending=True)

print('Release Point by Pitch Type (RHP):')
print(release_avg)

In [ ]:
# Release point scatter
fig, ax = plt.subplots(figsize=(10, 8))

pitch_types = release_avg['pitch_type_code'].to_list()[:7]
colors = plt.cm.Set1(np.linspace(0, 1, len(pitch_types)))

for i, ptype in enumerate(pitch_types):
    type_data = release_df.filter(pl.col('pitch_type_code') == ptype).sample(
        min(2000, len(release_df.filter(pl.col('pitch_type_code') == ptype))), seed=42
    )
    ax.scatter(
        type_data['x0'].to_list(),
        type_data['z0'].to_list(),
        alpha=0.3,
        s=10,
        label=ptype,
        color=colors[i]
    )

ax.set_xlabel('Horizontal Release (x0, feet)')
ax.set_ylabel('Vertical Release (z0, feet)')
ax.set_title('Release Point by Pitch Type (RHP)')
ax.legend()
plt.tight_layout()
plt.show()

---
## 6. Phase 5: Feature Importance & Correlation <a id='phase5'></a>

### 5.1 Correlation Analysis

In [ ]:
# Prepare features for correlation analysis
# Add all numeric features
feature_cols = [
    'inning', 'outs', 'away_score', 'home_score',
    'pitch_number', 'pitch_start_speed', 'pitch_end_speed',
    'px', 'pz', 'pfxX', 'pfxZ',
    'break_angle', 'break_length', 'break_vertical', 'break_horizontal',
    'x0', 'z0'
]

# Filter to available columns
available_features = [c for c in feature_cols if c in df.columns]

# Create correlation matrix
corr_df = df.select(available_features).to_pandas()
corr_matrix = corr_df.corr()

# Visualize
fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=ax, square=True, linewidths=0.5)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

### 5.2 Feature Importance for Pitch Type

In [ ]:
# Prepare data for mutual information
# Sample for computational efficiency
sample_df = df.sample(min(100000, len(df)), seed=42)

# Create feature matrix
feature_cols_mi = [
    'inning', 'outs', 'pitch_number',
    'pitch_start_speed', 'pfxX', 'pfxZ',
    'x0', 'z0', 'break_vertical', 'break_horizontal'
]

# Filter to rows with all features present
mi_df = sample_df.select(feature_cols_mi + ['pitch_type_code']).drop_nulls()

X = mi_df.select(feature_cols_mi).to_numpy()
y = LabelEncoder().fit_transform(mi_df['pitch_type_code'].to_list())

print(f'Samples for MI calculation: {len(X):,}')

In [ ]:
# Calculate mutual information for pitch type
mi_scores = mutual_info_classif(X, y, random_state=42)

mi_results = pd.DataFrame({
    'feature': feature_cols_mi,
    'mi_score': mi_scores
}).sort_values('mi_score', ascending=False)

print('Mutual Information Scores for Pitch Type Prediction:')
print(mi_results)

In [ ]:
# Visualize feature importance
fig, ax = plt.subplots(figsize=(10, 6))

ax.barh(mi_results['feature'], mi_results['mi_score'])
ax.set_xlabel('Mutual Information Score')
ax.set_title('Feature Importance for Pitch Type Prediction')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

### 5.3 Feature Importance for Location

In [ ]:
# Mutual information for location (px)
loc_feature_cols = [
    'inning', 'outs', 'pitch_number',
    'pitch_start_speed', 'pfxX', 'pfxZ',
    'x0', 'z0'
]

loc_mi_df = sample_df.select(loc_feature_cols + ['px', 'pz']).drop_nulls()
X_loc = loc_mi_df.select(loc_feature_cols).to_numpy()
y_px = loc_mi_df['px'].to_numpy()
y_pz = loc_mi_df['pz'].to_numpy()

# MI for px
mi_px = mutual_info_regression(X_loc, y_px, random_state=42)
mi_pz = mutual_info_regression(X_loc, y_pz, random_state=42)

loc_mi_results = pd.DataFrame({
    'feature': loc_feature_cols,
    'mi_px': mi_px,
    'mi_pz': mi_pz,
    'mi_combined': (mi_px + mi_pz) / 2
}).sort_values('mi_combined', ascending=False)

print('Mutual Information Scores for Location Prediction:')
print(loc_mi_results)

In [ ]:
# Visualize location feature importance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# px
axes[0].barh(loc_mi_results['feature'], loc_mi_results['mi_px'])
axes[0].set_xlabel('Mutual Information Score')
axes[0].set_title('Feature Importance for Horizontal Location (px)')
axes[0].invert_yaxis()

# pz
axes[1].barh(loc_mi_results['feature'], loc_mi_results['mi_pz'])
axes[1].set_xlabel('Mutual Information Score')
axes[1].set_title('Feature Importance for Vertical Location (pz)')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

---
## 7. Conclusions & Recommendations <a id='conclusions'></a>

### Key Findings

#### 1. Class Imbalance
- Fastballs (FF) dominate at ~30-35% of all pitches
- Sliders and Changeups are next most common (~10-15% each)
- Rare pitch types (KN, FS, KC) < 5% each
- **Recommendation**: Use class weighting or oversampling for rare pitch types

#### 2. Count Effects
- Strong relationship between count and pitch selection
- 0-2 counts favor breaking balls (waste pitches)
- 3-0/3-1 counts favor fastballs in the zone
- **Recommendation**: Count features (balls, strikes, pitcher_ahead, hitters_count) are critical

#### 3. Platoon Matchups
- Clear location differences by pitcher/batter handedness
- Same-side matchups favor away locations
- **Recommendation**: Include platoon_same_side feature

#### 4. Sequencing
- Strong pitch-to-pitch dependencies
- FF often follows FF
- Breaking balls often follow fastballs
- **Recommendation**: Previous pitch type is highly predictive

#### 5. Physical Metrics
- Velocity is highly predictive of pitch type
- Movement (pfxX, pfxZ) differentiates pitch types
- Release point varies by pitch type
- **Recommendation**: Consider adding velocity deltas and movement features

### Feature Recommendations

#### High Priority (Currently Used)
1. Count (balls, strikes)
2. Previous pitch type
3. Previous pitch location
4. Pitcher/batter IDs (embeddings)
5. Handedness (platoon matchup)

#### High Priority (Not Currently Used)
1. Velocity delta from previous pitch
2. Movement metrics (pfxX, pfxZ) of previous pitch
3. Pitcher's pitch mix (repertoire features)
4. At-bat pitch sequence (beyond just previous pitch)
5. Release point consistency

#### Medium Priority
1. Game context (score differential, inning)
2. Runners on base state
3. Pitcher fatigue (innings pitched)
4. Recent performance (last 10-20 pitches)

#### Low Priority (May Not Help Much)
1. Weather conditions
2. Ballpark factors
3. Time of day

In [ ]:
print('EDA Complete!')
print('\nSummary:')
print(f'- Analyzed {len(df):,} pitches from 2024 season')
print(f'- Multi-season analysis covered {len(multi_season_df):,} total pitches')
print(f'- Identified {len(mi_results)} features for pitch type prediction')
print(f'- Identified {len(loc_mi_results)} features for location prediction')